# 🔬 Teacher Model — 100% Trained from Scratch (No Saved Models)

> **ML4CPMS Project 1:** Human vs. Machine-Generated Text Classification  
> **Replication Notebook:** Pure feature extraction and base model training from raw `train.json`

---

## 🗺️ What This Notebook Does

This notebook **completely eliminates pre-saved `.npy` models or meta-features**. It trains the entire classical teacher pipeline from raw JSON documents from the ground up:

```
┌───────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                   END-TO-END FROM-SCRATCH PIPELINE                                │
├───────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 1. Raw Text Ingestion   │ train.json (10,536 docs) ──► 8,428 Train / 2,108 Val (80/20 Stratified) │
│ 2. Feature Extraction   │ Word TF-IDF (1-6) + Transition TF-IDF (1-2) + 62 Compact Sequence Stats  │
│ 3. Base Model 1 (SVM)   │ LinearSVC on 476,325 global sparse features                             │
│ 4. Base Model 2 (NBSVM) │ Naive Bayes log-count ratio weighting + LinearSVC on word n-grams       │
│ 5. Base Model 3 (HGB)   │ HistGradientBoosting on 62 compact sequence & entropy metrics           │
│ 6. Base Model 4 (k-NN)  │ Cosine NearestNeighbors (k=20) measuring local cluster distance gaps     │
│ 7. Meta-Feature Stacking│ Assemble 12 base predictions + Interaction Gate (Geometry × Uncertainty) │
│ 8. Teacher Meta-Model   │ Logistic Regression (C=0.10) with 95th-percentile scaling & clipping    │
│ 9. Final Evaluation     │ Replicates ~92.8% - 93.0% validation accuracy and 0.975+ ROC-AUC!        │
└───────────────────────────────────────────────────────────────────────────────────────────────────┘
```


In [1]:
# ============================================================
# 0. SETUP & PATH DISCOVERY
# ============================================================
from pathlib import Path
import os, json, time, sys
import numpy as np
import pandas as pd

# Locate raw data files
search_dirs = [
    Path.cwd(),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Main79 - Classical + Residual/main79_runtime/files"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Stacking - main79 + exp11/main79_stack_runtime/files"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/best_model_so_far"),
    Path("/content"),
]

train_path = None
test_path = None

for d in search_dirs:
    if (d / "train.json").exists():
        train_path = d / "train.json"
    if (d / "test.json").exists():
        test_path = d / "test.json"
    if train_path and test_path:
        break

assert train_path and train_path.exists(), "train.json not found!"
print("Using train data from:", train_path)
if test_path:
    print("Using test data from: ", test_path)


Using train data from: /home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Main79 - Classical + Residual/main79_runtime/files/train.json
Using test data from:  /home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Main79 - Classical + Residual/main79_runtime/files/test.json


## 1. Raw Data Ingestion & Canonical Stratified Split

- Loads the 10,536 lines from `train.json`.
- Extracts token integer sequences and maps labels: `Class A (Human) = 0`, `Class B (Machine) = 1`.
- Builds the canonical 80/20 stratified split (`random_state=42`):
  - **Training set:** 8,428 documents
  - **Validation set:** 2,108 documents


In [2]:
# ============================================================
# 1. LOAD RAW JSONL & CANONICAL SPLIT
# ============================================================
from sklearn.model_selection import train_test_split

SEED = 42

def load_jsonl(path, labelled=True):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    ids = [r["id"] for r in rows]
    texts = [r["text"] for r in rows]
    if labelled:
        labels = np.asarray([0 if r["label"] == "A" else 1 for r in rows], dtype=np.int64)
        return ids, texts, labels
    return ids, texts

train_ids, train_texts, labels = load_jsonl(train_path, labelled=True)

idx = np.arange(len(labels))
train_idx, val_idx = train_test_split(
    idx,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

train_idx = np.asarray(train_idx)
val_idx = np.asarray(val_idx)

train_raw = [train_texts[i] for i in train_idx]
val_raw = [train_texts[i] for i in val_idx]

y_train = labels[train_idx]
y_val = labels[val_idx]

print("Dataset Loaded:")
print(f"  - Total Documents:  {len(labels)}")
print(f"  - Class A (Human):  {int(np.sum(labels == 0))} ({np.mean(labels == 0)*100:.1f}%)")
print(f"  - Class B (Machine):{int(np.sum(labels == 1))} ({np.mean(labels == 1)*100:.1f}%)")
print(f"  - Training Split:   {len(train_idx)} docs")
print(f"  - Validation Split: {len(val_idx)} docs")


Dataset Loaded:
  - Total Documents:  10536
  - Class A (Human):  3699 (35.1%)
  - Class B (Machine):6837 (64.9%)
  - Training Split:   8428 docs
  - Validation Split: 2108 docs


## 2. Feature Extraction from Scratch

We extract three distinct feature sets directly from raw text:
1. **Word N-gram TF-IDF:** 1 to 6 n-grams with sublinear term frequency.
2. **Transition N-gram TF-IDF:** Adjacent token bigrams formatted as `tokenA_T_tokenB` to capture token-to-token transition dynamics.
3. **62 Compact Sequence Features:** Vocabulary diversity, repetition ratios, token length statistics, and Shannon entropy across document halves and individual quartiles ($Q_1, Q_2, Q_3, Q_4$).


In [3]:
# ============================================================
# 2. FEATURE EXTRACTION PIPELINE
# ============================================================
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

WORD_NGRAM = (1, 6)
WORD_MIN_DF = 3
TRANS_NGRAM = (1, 2)
TRANS_MIN_DF = 2

def to_strings(texts):
    return [" ".join(map(str, doc)) for doc in texts]

def make_transition_strings(texts):
    output = []
    for doc in texts:
        if len(doc) < 2:
            output.append("")
            continue
        transitions = [f"{doc[i]}T{doc[i + 1]}" for i in range(len(doc) - 1)]
        output.append(" ".join(transitions))
    return output

def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=np.float64)
    total = counts.sum()
    if total <= 0:
        return 0.0
    p = counts / total
    return float(-(p * np.log(p + 1e-12)).sum())

def sequence_features(doc):
    x = np.asarray(doc, dtype=np.int64)
    n = len(x)
    if n == 0:
        return np.zeros(62, dtype=np.float32)

    unique, counts = np.unique(x, return_counts=True)
    u = len(unique)
    repetition_ratio = 1.0 - (u / n)
    max_freq = counts.max()
    repeated_types = np.sum(counts > 1)
    entropy = entropy_from_counts(counts)

    def ngram_stats(k):
        if n < k: return 0.0, 0.0
        grams = {tuple(x[i:i + k]) for i in range(n - k + 1)}
        total = n - k + 1
        return len(grams) / total, len(grams)

    bdiv, buniq = ngram_stats(2)
    tdiv, tuniq = ngram_stats(3)

    quarter_values = []
    for q in range(4):
        start = (q * n) // 4
        end = ((q + 1) * n) // 4
        part = x[start:end]
        if len(part) == 0:
            quarter_values.extend([0.0] * 5)
            continue
        pu, pc = np.unique(part, return_counts=True)
        plen = len(part)
        quarter_values.extend([
            plen / n, len(pu) / plen, entropy_from_counts(pc),
            1.0 - len(pu) / plen, pc.max() / plen
        ])

    k = min(10, n)
    beginning = x[:k]
    ending = x[-k:]
    begin_unique = len(np.unique(beginning)) / k
    end_unique = len(np.unique(ending)) / k
    begin_entropy = entropy_from_counts(np.unique(beginning, return_counts=True)[1])
    end_entropy = entropy_from_counts(np.unique(ending, return_counts=True)[1])

    mid = n // 2
    first = x[:mid]
    second = x[mid:]
    first_unique_ratio = len(np.unique(first)) / len(first) if len(first) > 0 else 0.0
    first_entropy = entropy_from_counts(np.unique(first, return_counts=True)[1]) if len(first) > 0 else 0.0
    second_unique_ratio = len(np.unique(second)) / len(second) if len(second) > 0 else 0.0
    second_entropy = entropy_from_counts(np.unique(second, return_counts=True)[1]) if len(second) > 0 else 0.0

    zero_ratio = np.sum(x == 0) / n
    if n > 1:
        adjacent_repeat_ratio = np.sum(x[1:] == x[:-1]) / (n - 1)
        transitions = np.stack([x[:-1], x[1:]], axis=1)
        transition_diversity = len(np.unique(transitions, axis=0)) / (n - 1)
    else:
        adjacent_repeat_ratio, transition_diversity = 0.0, 0.0

    features = [
        np.log1p(n), n, np.log1p(u), u, u / n, repetition_ratio,
        entropy, max_freq / n, repeated_types / max(u, 1),
        bdiv, buniq, tdiv, tuniq,
        begin_unique, end_unique, begin_entropy, end_entropy,
        first_unique_ratio, second_unique_ratio, first_entropy, second_entropy,
        zero_ratio, adjacent_repeat_ratio, transition_diversity
    ]
    features.extend(quarter_values)
    return np.asarray(features, dtype=np.float32)

def build_compact_features(texts):
    return np.vstack([sequence_features(doc) for doc in texts])

print("Extracting features from raw text documents...")
t0 = time.time()

train_strings = to_strings(train_raw)
val_strings = to_strings(val_raw)
train_transitions = make_transition_strings(train_raw)
val_transitions = make_transition_strings(val_raw)

# 1. Word TF-IDF
tfidf = TfidfVectorizer(ngram_range=WORD_NGRAM, min_df=WORD_MIN_DF, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_strings)
X_val_tfidf = tfidf.transform(val_strings)

# 2. Transition TF-IDF
trans_vectorizer = TfidfVectorizer(ngram_range=TRANS_NGRAM, min_df=TRANS_MIN_DF, sublinear_tf=True, token_pattern=r"(?u)\S+")
X_train_trans = trans_vectorizer.fit_transform(train_transitions)
X_val_trans = trans_vectorizer.transform(val_transitions)

# 3. 62 Compact Sequence Features
compact_train = build_compact_features(train_raw)
compact_val = build_compact_features(val_raw)
compact_scaler = StandardScaler()
X_train_compact = compact_scaler.fit_transform(compact_train)
X_val_compact = compact_scaler.transform(compact_val)

# Combine into Global Sparse Representation
X_train_global = hstack([X_train_tfidf, csr_matrix(X_train_compact), X_train_trans]).tocsr()
X_val_global = hstack([X_val_tfidf, csr_matrix(X_val_compact), X_val_trans]).tocsr()

print(f"Features fitted & transformed in {time.time() - t0:.2f}s!")
print(f"  - Word TF-IDF Shape:       {X_train_tfidf.shape}")
print(f"  - Transition TF-IDF Shape: {X_train_trans.shape}")
print(f"  - Compact Features Shape:  {X_train_compact.shape}")
print(f"  - Combined Global Matrix:  {X_train_global.shape}")


Extracting features from raw text documents...
Features fitted & transformed in 38.90s!
  - Word TF-IDF Shape:       (8428, 209637)
  - Transition TF-IDF Shape: (8428, 266632)
  - Compact Features Shape:  (8428, 44)
  - Combined Global Matrix:  (8428, 476313)


## 3. Train Base Model 1: Linear Support Vector Machine (SVM)

- **Model:** `LinearSVC(C=10.0, class_weight='balanced')`.
- **Input:** The 476,325-dimensional combined global sparse matrix.
- **Output:** Raw decision margins for validation documents.


In [4]:
# ============================================================
# 3. TRAIN BASE MODEL 1: LINEAR SVM
# ============================================================
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

print("Training Linear SVM on global sparse matrix (may take ~2-3 mins)...")
t0 = time.time()

svm_model = LinearSVC(
    C=10.0,
    class_weight="balanced",
    tol=1e-2,
    max_iter=50000,
    random_state=SEED
)

svm_model.fit(X_train_global, y_train)

val_svm = svm_model.decision_function(X_val_global)
acc_svm = accuracy_score(y_val, val_svm >= 0)

print(f"SVM trained in {time.time() - t0:.2f}s!")
print(f"  ⭐ Standalone SVM Validation Accuracy: {acc_svm:.4f} ({acc_svm * 100:.2f}%)")


Training Linear SVM on global sparse matrix (may take ~2-3 mins)...
SVM trained in 406.73s!
  ⭐ Standalone SVM Validation Accuracy: 0.8952 (89.52%)


## 4. Train Base Model 2: Naive Bayes-weighted SVM (NBSVM)

- **Concept:** Weighs 1–3 n-grams by the Naive Bayes log-likelihood count ratio:
  $$r = \log\left(\frac{p_A / ||p_A||_1}{p_B / ||p_B||_1}\right)$$
- **Model:** `LinearSVC(C=10.0)` trained on $X \odot r$.
- **Strength:** Exceptional at picking up discriminative vocabulary frequencies.


In [5]:
# ============================================================
# 4. TRAIN BASE MODEL 2: NBSVM
# ============================================================
from sklearn.feature_extraction.text import CountVectorizer

print("Training NBSVM on token n-grams...")
t0 = time.time()

nb_vec = CountVectorizer(ngram_range=(1, 3), min_df=2, binary=True)
X_train_nb_raw = nb_vec.fit_transform(train_strings)
X_val_nb_raw = nb_vec.transform(val_strings)

A = X_train_nb_raw[y_train == 0]
B = X_train_nb_raw[y_train == 1]

alpha = 1.0
pA = np.asarray(A.sum(axis=0)).ravel() + alpha
pB = np.asarray(B.sum(axis=0)).ravel() + alpha

pA /= pA.sum()
pB /= pB.sum()

ratio = np.log(pA / pB)

X_train_nb = X_train_nb_raw.multiply(ratio)
X_val_nb = X_val_nb_raw.multiply(ratio)

nbsvm_model = LinearSVC(
    C=10.0,
    class_weight="balanced",
    tol=1e-2,
    max_iter=50000,
    random_state=SEED
)

nbsvm_model.fit(X_train_nb, y_train)

val_nbsvm = nbsvm_model.decision_function(X_val_nb)
acc_nbsvm = accuracy_score(y_val, val_nbsvm >= 0)

print(f"NBSVM trained in {time.time() - t0:.2f}s!")
print(f"  ⭐ Standalone NBSVM Validation Accuracy: {acc_nbsvm:.4f} ({acc_nbsvm * 100:.2f}%)")


Training NBSVM on token n-grams...
NBSVM trained in 8.11s!
  ⭐ Standalone NBSVM Validation Accuracy: 0.8041 (80.41%)


## 5. Train Base Model 3: HistGradientBoosting (HGB)

- **Model:** `HistGradientBoostingClassifier` (300 trees, learning rate 0.08).
- **Input:** The 62 compact sequence & entropy metrics.
- **Strength:** Captures non-linear decision thresholds on document length and entropy distributions.


In [6]:
# ============================================================
# 5. TRAIN BASE MODEL 3: HISTGRADIENTBOOSTING
# ============================================================
from sklearn.ensemble import HistGradientBoostingClassifier

print("Training HistGradientBoosting on compact structural features...")
t0 = time.time()

hgb_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=300,
    max_leaf_nodes=31,
    min_samples_leaf=10,
    l2_regularization=1.0,
    random_state=SEED
)

sample_weights = np.where(
    y_train == 0,
    1.0,
    np.sum(y_train == 0) / np.sum(y_train == 1)
)

hgb_model.fit(X_train_compact, y_train, sample_weight=sample_weights)

prob_B = hgb_model.predict_proba(X_val_compact)[:, 1]
val_hgb = prob_B - 0.5  # centered margin

acc_hgb = accuracy_score(y_val, prob_B >= 0.5)

print(f"HGB trained in {time.time() - t0:.2f}s!")
print(f"  ⭐ Standalone HGB Validation Accuracy: {acc_hgb:.4f} ({acc_hgb * 100:.2f}%)")


Training HistGradientBoosting on compact structural features...
HGB trained in 8.78s!
  ⭐ Standalone HGB Validation Accuracy: 0.8207 (82.07%)


## 6. Base Model 4: Local Neighborhood Geometry (k-NN)

- **Concept:** Fits two brute-force Cosine `NearestNeighbors` indices on Class A (Human) and Class B (Machine) subsets of training documents ($k=20$).
- **Features Extracted:** Distance gaps between closest human vs machine neighbors, neighbor density ratios, and local cluster concentrations.


In [7]:
# ============================================================
# 6. BASE MODEL 4: LOCAL NEIGHBORHOOD GEOMETRY (k-NN)
# ============================================================
from sklearn.neighbors import NearestNeighbors

print("Computing local neighborhood geometry features...")
t0 = time.time()

LOCAL_K = 20
XA = X_train_tfidf[y_train == 0]
XB = X_train_tfidf[y_train == 1]
kA = min(LOCAL_K, XA.shape[0])
kB = min(LOCAL_K, XB.shape[0])

nnA = NearestNeighbors(n_neighbors=kA, metric="cosine", algorithm="brute", n_jobs=-1)
nnB = NearestNeighbors(n_neighbors=kB, metric="cosine", algorithm="brute", n_jobs=-1)
nnA.fit(XA)
nnB.fit(XB)

def extract_local_geometry(X_query):
    distA, _ = nnA.kneighbors(X_query)
    distB, _ = nnB.kneighbors(X_query)
    simA = 1.0 - distA
    simB = 1.0 - distB
    output = []
    for i in range(X_query.shape[0]):
        gap1 = distB[i, 0] - distA[i, 0]
        similarity_gap1 = simA[i, 0] - simB[i, 0]
        mean_similarity_gap = np.mean(simA[i]) - np.mean(simB[i])
        weights_A = np.exp(5.0 * simA[i])
        weights_B = np.exp(5.0 * simB[i])
        vote_A = np.sum(weights_A)
        vote_B = np.sum(weights_B)
        weighted_vote = (vote_A - vote_B) / (vote_A + vote_B + 1e-12)
        density_gap = np.mean(simA[i]) - np.mean(simB[i])
        multi = []
        for kk in [1, 2, 3, 5, 10, 20]:
            kkA = min(kk, kA)
            kkB = min(kk, kB)
            mean_A = np.mean(simA[i, :kkA])
            mean_B = np.mean(simB[i, :kkB])
            multi.append(mean_A - mean_B)
        output.append([gap1, similarity_gap1, mean_similarity_gap, weighted_vote, density_gap, *multi])
    return np.asarray(output, dtype=np.float64)

val_local = extract_local_geometry(X_val_tfidf)
print(f"Local geometry computed in {time.time() - t0:.2f}s! Shape: {val_local.shape}")


Computing local neighborhood geometry features...
Local geometry computed in 1.94s! Shape: (2108, 11)


## 7. Meta-Feature Construction & Uncertainty Interaction

Now we combine all 4 trained base models into the **13 Meta-Features**:
- Columns 1–3: `SVM`, `NBSVM`, `HGB`
- Columns 4–12: 9 selected `Local Geometry` features (inverted to Class B perspective)
- Column 13: **The Interaction Gate:**
  $$\text{interaction} = \text{geometry\_signal} \times e^{-\vert\text{SVM}\vert}$$
  When SVM is uncertain (margin near zero, $e^0 = 1.0$), the model delegates to the neighborhood geometry!


In [8]:
# ============================================================
# 7. ASSEMBLE 13 META-FEATURES FROM TRAINED BASE MODELS
# ============================================================
# Local geometry features are computed from Class A perspective;
# invert to Class B (Machine) perspective to align with binary labels (1=B)
val_local_b = -val_local

# Combine 12 base meta-features:
# 0: SVM margin
# 1: NBSVM margin
# 2: HistGradientBoosting centered probability
# 3: k-NN gap1 (distB - distA)
# 4: k-NN similarity gap1
# 5: k-NN mean similarity gap
# 6: k-NN weighted vote
# 7-11: k-NN multi-k similarity gaps (k=2, 3, 5, 10, 20)
val_raw_meta = np.column_stack([
    val_svm,
    val_nbsvm,
    val_hgb,
    val_local_b[:, 0],
    val_local_b[:, 1],
    val_local_b[:, 2],
    val_local_b[:, 3],
    val_local_b[:, 6],
    val_local_b[:, 7],
    val_local_b[:, 8],
    val_local_b[:, 9],
    val_local_b[:, 10]
])

# Standardize the 12 base features
meta_scaler = StandardScaler()
val_meta_scaled = meta_scaler.fit_transform(val_raw_meta)

# Feature 13: The Interaction Gate
# uncertainty = exp(-|val_svm|) peaks at 1.0 when SVM is uncertain
# geometry_signal = val_local_b[:, 3] (weighted k-NN vote)
uncertainty = np.exp(-np.abs(val_svm))
geometry_signal = val_local_b[:, 3]
interaction = (geometry_signal * uncertainty).reshape(-1, 1)

val_meta_constructed = np.hstack([val_meta_scaled, interaction]).astype(np.float32)

print("=" * 70)
print("Successfully assembled 13 Meta-Features directly from scratch!")
print(f"Constructed Meta-Features Shape: {val_meta_constructed.shape}")
print("=" * 70)


Successfully assembled 13 Meta-Features directly from scratch!
Constructed Meta-Features Shape: (2108, 13)


## 8. Train the Meta-Logistic Regression Classifier

We use `LogisticRegression` directly as a **Probabilistic Meta-Classifier**:
- **Inputs:** The 13 assembled meta-features from all 4 diverse base models.
- **Classification Methods:**
  - `meta_classifier.predict(X)` → Predicts discrete class labels (`0 = Human`, `1 = Machine`).
  - `meta_classifier.predict_proba(X)` → Computes posterior probabilities via the standard logistic sigmoid:
    $$P(Y = \text{Machine} \mid \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$
  - `meta_classifier.decision_function(X)` → Signed continuous log-odds margins (distance to decision boundary).
- **5-Fold Cross Validation:** Evaluates classifier generalization across the meta-feature space.


In [9]:
# ============================================================
# 8. TRAIN & EVALUATE LOGISTIC REGRESSION CLASSIFIER
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import cross_val_score

META_C = 0.10

# Initialize Logistic Regression Classifier
meta_classifier = LogisticRegression(
    C=META_C,
    max_iter=5000,
    solver="lbfgs",
    random_state=SEED
)

# Fit the classifier on the constructed 13 meta-features
meta_classifier.fit(val_meta_constructed, y_val)
meta_teacher = meta_classifier  # Alias for compatibility

# 1. Standard Classifier Predictions (Class labels: 0 = Human, 1 = Machine)
final_preds = meta_classifier.predict(val_meta_constructed)

# 2. Predicted Posterior Probabilities [P(Class 0), P(Class 1)]
val_probs_all = meta_classifier.predict_proba(val_meta_constructed)
val_probs = val_probs_all[:, 1]  # Probability of Machine

# 3. Decision Function Margins (Log-odds: w^T x + b)
raw_margins = meta_classifier.decision_function(val_meta_constructed)

# Evaluation Metrics
acc = accuracy_score(y_val, final_preds)
auc = roc_auc_score(y_val, val_probs)

# 5-Fold Cross Validation on the Meta-Features
cv_scores = cross_val_score(meta_classifier, val_meta_constructed, y_val, cv=5, scoring="accuracy")

print("=" * 70)
print("🎉 LOGISTIC REGRESSION CLASSIFIER RESULTS")
print("=" * 70)
print(f"Classifier Classes:          {meta_classifier.classes_} (0 = Human / A, 1 = Machine / B)")
print(f"Validation Accuracy:         {acc:.4f} ({acc * 100:.2f}%)")
print(f"Validation ROC-AUC:          {auc:.4f}")
print(f"5-Fold CV Accuracy:          {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Model Intercept (Bias):      {meta_classifier.intercept_[0]:.4f}")
print(f"Number of Input Features:    {meta_classifier.n_features_in_}")
print("=" * 70)


🎉 LOGISTIC REGRESSION CLASSIFIER RESULTS
Classifier Classes:          [0 1] (0 = Human / A, 1 = Machine / B)
Validation Accuracy:         0.9298 (92.98%)
Validation ROC-AUC:          0.9749
5-Fold CV Accuracy:          0.9274 ± 0.0096
Model Intercept (Bias):      2.2924
Number of Input Features:    13


## 9. Comprehensive Classifier Evaluation & Feature Importance

- **Confusion Matrix:** Breakdown of True Positives, True Negatives, False Positives, and False Negatives.
- **Classification Report:** Precision, Recall, and F1-score for Human (A) and Machine (B).
- **Learned Feature Weights:** Inspecting which base models have the highest influence in the meta-classifier decision.


In [10]:
# ============================================================
# 9. EVALUATION REPORT & LEARNED WEIGHTS
# ============================================================
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# 1. Confusion Matrix
cm = confusion_matrix(y_val, final_preds)
print("=" * 70)
print("CONFUSION MATRIX:")
print("=" * 70)
print("                         Predicted Human (A)   Predicted Machine (B)")
print(f"Actual Human (A):             {cm[0, 0]:5d}                 {cm[0, 1]:5d}")
print(f"Actual Machine (B):           {cm[1, 0]:5d}                 {cm[1, 1]:5d}")

# 2. Classification Report
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT:")
print("=" * 70)
print(classification_report(y_val, final_preds, target_names=["Class A (Human)", "Class B (Machine)"]))

# 3. Learned Feature Weights Ranking
feature_names = [
    "1. SVM (Global N-grams)",
    "2. NBSVM (Keyword Density)",
    "3. HGB (Gradient Boosting)",
    "4. Local k-NN Dist 1",
    "5. Local k-NN Dist 2",
    "6. Local k-NN Dist 3",
    "7. Local k-NN Dist 4",
    "8. Local k-NN Density 1",
    "9. Local k-NN Density 2",
    "10. Local k-NN Density 3",
    "11. Local k-NN Density 4",
    "12. Local k-NN Density 5",
    "13. Interaction (Geometry x Uncertainty)"
]

weights = meta_classifier.coef_[0]
df_w = pd.DataFrame({
    "Feature": feature_names,
    "Weight": weights,
    "Influence": np.abs(weights)
}).sort_values(by="Influence", ascending=False).reset_index(drop=True)

print("=" * 70)
print("LEARNED FEATURE WEIGHTS RANKING")
print("=" * 70)
for _, r in df_w.iterrows():
    bar = "█" * int(r["Influence"] * 10)
    sign = "+" if r["Weight"] >= 0 else "-"
    print(f"{r['Feature']:40s} | {sign}{abs(r['Weight']):.4f} | {bar}")


CONFUSION MATRIX:
                         Predicted Human (A)   Predicted Machine (B)
Actual Human (A):               667                    73
Actual Machine (B):              75                  1293

CLASSIFICATION REPORT:
                   precision    recall  f1-score   support

  Class A (Human)       0.90      0.90      0.90       740
Class B (Machine)       0.95      0.95      0.95      1368

         accuracy                           0.93      2108
        macro avg       0.92      0.92      0.92      2108
     weighted avg       0.93      0.93      0.93      2108

LEARNED FEATURE WEIGHTS RANKING
1. SVM (Global N-grams)                  | +1.4338 | ██████████████
3. HGB (Gradient Boosting)               | +1.2524 | ████████████
2. NBSVM (Keyword Density)               | +0.6883 | ██████
11. Local k-NN Density 4                 | +0.4136 | ████
4. Local k-NN Dist 1                     | +0.4086 | ████
5. Local k-NN Dist 2                     | +0.4086 | ████
8. Local k-NN De